### Lab Instructions

Pick one feature set and run all of the code cells in order. Answer questions in your own words in markdown cells. Download as html to submit.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn import preprocessing
from sklearn.decomposition import PCA
from sklearn.utils import resample
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn import svm
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import confusion_matrix
import scipy.stats

import warnings
warnings.filterwarnings('ignore')

In [ ]:
data = pd.read_csv("daic_woz_mild_depression_subset.csv") #features for the Moodable/EMU dataset
#use if there are extra columns
#data = data.drop(columns = 'id')
print(data.shape)
data.head()

In [ ]:
# Set this to the actual column name in your dataset
target_col = "label"

# Create binary labels specifically for mild depression (scores 5-9)
data[target_col] = ((data['PHQ8_Score'] >= 5) & (data['PHQ8_Score'] <= 9)).astype(int)

# checking values
data[target_col].value_counts() / data.shape[0]

## Machine learning code with comments added for you

In [ ]:
#reading about python def https://www.w3schools.com/python/ref_keyword_def.asp

def get_ba(x,y,z,q):
    #fit model and make predictions
    clf.fit(x, list(y))
    y_pred = clf.predict(z)

    #evaluate model - testing
    conf_mat = confusion_matrix(list(q), y_pred)
    TN = conf_mat[0][0]
    TP = conf_mat[1][1]
    FP = conf_mat[0][1]
    FN = conf_mat[1][0]
    # add sensitivity, specificity, and ba here

    accuracy = (TP+TN)/(TP+TN+FP+FN)

    return accuracy #change to return ba instead of accuracy

In [ ]:

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn import preprocessing
from sklearn.decomposition import PCA
from sklearn.utils import resample
from sklearn.metrics import confusion_matrix
from sklearn.naive_bayes import GaussianNB
from sklearn import svm
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from xgboost import XGBClassifier
from sklearn.feature_extraction.text import TfidfVectorizer


label = "label" # Ensure this matches your target column name exactly

numberOfFeatures = 2 #set number of principal components
modelTypelist = ["RF", "NB", "kNN", "LR", "SVC1", "SVC2", "AdaBoost", "XGBoost"]

train_ba_depression = []
test_ba_depression = []
train_sens_depression = []
test_sens_depression = []
train_spec_depression = []
test_spec_depression = []

rlist = []
mlist = []


def get_ba(x, y, z, q):
    # fit model and make predictions
    clf.fit(x, list(y))
    y_pred = clf.predict(z)

    # evaluate model - testing
    conf_mat = confusion_matrix(list(q), y_pred)
    TN = conf_mat[0][0]
    TP = conf_mat[1][1]
    FP = conf_mat[0][1]
    FN = conf_mat[1][0]

    # Calculate sensitivity, specificity, and balanced accuracy
    sensitivity = TP / (TP + FN) if (TP + FN) != 0 else 0
    specificity = TN / (TN + FP) if (TN + FP) != 0 else 0
    ba = (sensitivity + specificity) / 2

    return ba, sensitivity, specificity


for modelType in modelTypelist:
    print("Working on", modelType)
    for r in range(50, 65): #runs 15 times
        rlist.append(r)
        mlist.append(modelType)

        #create train/test sets
        df_train, df_test = train_test_split(data, test_size=0.2, stratify=data[[label]], random_state = r)

        #save target variables
        train_targets = df_train.loc[:,[label]]
        test_targets = df_test.loc[:,[label]]


        # The scaler and models only take numbers, so we convert the 'text' column into numeric features
        vectorizer = TfidfVectorizer(max_features=1000, stop_words='english')

        # Isolate the text column and fill any blanks
        train_text = df_train['text'].fillna('')
        test_text = df_test['text'].fillna('')

        # Transform the string text into numeric arrays
        trainContent = vectorizer.fit_transform(train_text).toarray()
        testContent = vectorizer.transform(test_text).toarray()

        #normalize before pca
        min_max_scaler = preprocessing.MinMaxScaler()
        np_scaled = min_max_scaler.fit_transform(trainContent)
        featureSubset = pd.DataFrame(np_scaled)
        np_scaled2 =  min_max_scaler.transform(testContent)
        testSubset = pd.DataFrame(np_scaled2)

        #principal component analysis (PCA Activated)
        pca = PCA(n_components=numberOfFeatures)
        pca = pca.fit(featureSubset)
        X_pca = pca.transform(featureSubset)
        pcaDF = pd.DataFrame(X_pca)
        testSubset2 = pca.transform(testSubset)
        pca_test = pd.DataFrame(testSubset2)

        #upsampling to balance classes
        train_targets = train_targets.reset_index(drop = True)

        #reattach labels using the PCA dataframe
        DF_labels = pd.concat([pcaDF, train_targets], axis = 1)

        phq0 = DF_labels[DF_labels[label] == 0]
        phq1 = DF_labels[DF_labels[label] == 1]

        if phq0.shape[0] > phq1.shape[0]:
            phq_upsampled = resample(phq1, n_samples=(phq0.shape[0]-phq1.shape[0]), random_state=50)
            phq_up = pd.concat([DF_labels, phq_upsampled])
        elif phq0.shape[0] < phq1.shape[0]:
            phq_upsampled = resample(phq0, n_samples=(phq1.shape[0]-phq0.shape[0]), random_state=50)
            phq_up = pd.concat([DF_labels, phq_upsampled])
        else:
            phq_up = DF_labels # In case they are already perfectly balanced

        #remove upsampled labels
        phq_targets = phq_up.loc[:,[label]]
        phq_features = phq_up.drop(columns = [label])

        #select modeltype
        if modelType == "NB":
            clf = GaussianNB()
        elif modelType == "SVC1":
            clf = svm.SVC(kernel='rbf', random_state=r)
        elif modelType == "SVC2":
            clf = svm.SVC(kernel='linear', random_state=r)
        elif modelType == "kNN":
            clf = KNeighborsClassifier()
        elif modelType == "LR":
            clf = LogisticRegression(random_state=r)
        elif modelType == "RF":
            clf = RandomForestClassifier(n_estimators = 10, random_state=r, max_depth = 5)
        elif modelType == "AdaBoost":
            clf = AdaBoostClassifier(random_state=r)
        elif modelType == "XGBoost":
            clf = XGBClassifier(random_state=r, max_depth=3)
        else:
            print("Error: check model selection code")

        #train evaluate models
        test_ba, test_sens, test_spec = get_ba(phq_features, phq_targets[label], pca_test, test_targets[label])
        test_ba_depression.append(test_ba)
        test_sens_depression.append(test_sens)
        test_spec_depression.append(test_spec)

        train_ba, train_sens, train_spec = get_ba(phq_features, phq_targets[label], phq_features, phq_targets[label])
        train_ba_depression.append(train_ba)
        train_sens_depression.append(train_sens)
        train_spec_depression.append(train_spec)

resultsDF = pd.DataFrame()
resultsDF["train_ba"] = train_ba_depression
resultsDF["test_ba"] = test_ba_depression
resultsDF["train_sens"] = train_sens_depression
resultsDF["test_sens"] = test_sens_depression
resultsDF["train_spec"] = train_spec_depression
resultsDF["test_spec"] = test_spec_depression
resultsDF["random"] = rlist
resultsDF["model"] = mlist

print("\n--- FINAL AVERAGES ---")
for modelType in modelTypelist:
    tempDF = resultsDF[resultsDF.model == modelType]
    print(f"[{modelType}]")
    print(f"  Balanced Accuracy: {np.mean(tempDF['test_ba']):.2f} +- {np.std(tempDF['test_ba']):.2f}")
    print(f"  Sensitivity:       {np.mean(tempDF['test_sens']):.2f} +- {np.std(tempDF['test_sens']):.2f}")
    print(f"  Specificity:       {np.mean(tempDF['test_spec']):.2f} +- {np.std(tempDF['test_spec']):.2f}\n")

# Display the first few rows of the final dataframe
display(resultsDF.head())

## Machine learning code updates

In the code cell above, add AdaBoost and XGBoost
* Add model names to modelTypelist
* Add elif condition with the same model names
* Add function to call models so clf equals those models
* Make sure the set max_depth as appropriate to mitigate overfitting. Can also adjust random forest max_depth.

Add sensitivity, specificity, and balanced accuracy to the get_ba function:
* Add equations for sensitity, specificity, and balanced accuracy
* Return balanced accuracy, sensitivity, and specificity
* Update line where get_ba is called to save all three metrics
* Add sensitivity and specificity as columns in the results dataframe
* In addition to ba, print out sensitivity and specificity averages below

You may choose to switch to PCA instead of raw features- especially if you select a high dimensional dataset.

## Plotting

In [ ]:
bas = []
for modelType in modelTypelist:
    tempDF = resultsDF[resultsDF.model == modelType]
    bas.append(tempDF['test_ba'].to_numpy())

In [ ]:
plt.boxplot(bas)
plt.xticks(range(0, len(bas)+1), [' ']+ modelTypelist)
plt.ylim(0, 1)
plt.show()

In [ ]:
for modelType in modelTypelist:
    tempDF = resultsDF[resultsDF.model == modelType]
    print(modelType + ": " + str(round(np.mean(tempDF["test_ba"]),2)) + " +- " + str(round(np.std(tempDF["test_ba"]),2)))

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Set the figure size for better readability
plt.figure(figsize=(12, 6))

# Create a boxplot using the results DataFrame
sns.boxplot(x='model', y='test_ba', data=resultsDF, palette='Set2')

# Add a swarmplot on top to see the individual data points (the 15 runs per model)
sns.swarmplot(x='model', y='test_ba', data=resultsDF, color=".25", alpha=0.6)

# Add titles and labels
plt.title('Test Balanced Accuracy Distribution Across ML Models (15 Runs)', fontsize=16)
plt.xlabel('Machine Learning Model', fontsize=12)
plt.ylabel('Test Balanced Accuracy (test_ba)', fontsize=12)

# Draw a horizontal line at 0.50 (baseline random guessing for balanced accuracy)
plt.axhline(y=0.50, color='r', linestyle='--', label='Baseline (0.50)')
plt.legend()

# Display the plot
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

 ## Questions to Answer

Make sure to refer to numeric output in your answers.

1. Which model is known for text classification. Did it peform the best?

2. Which models are most at risk of overfitting? Did they peform better than the other models?

3. Which models perform feature selection or regularization? Did they perform better than the other models?

4. Which models are parameteric? Did they peform better than the other models?

5. What model performed best? How stable was it? What does the sensitivity and specificity indicate?

6. How does the best model and its peformance differ across group member's tasks? Interpret.

1. Which model is known for text classification? Did it perform the best?

Naive Bayes (NB) is usually the go-to for text classification, but it actually wasn't the best one here. It only got a balanced accuracy of 0.54, losing out to Logistic Regression (LR) which hit 0.63. Also, even though NB's sensitivity (0.75) and specificity (0.32) looked a bit better than before, it still has a bad habit of over-predicting the positive class. Basically, it really struggles to pick out the negative cases accurately.

2. Which models are most at risk of overfitting? Did they perform better than the other models?

Tree-based models like Random Forest, AdaBoost, and XGBoost are definitely the most likely to overfit. They actually didn't do as well as LR or the SVCs. AdaBoost got a 0.58, XGBoost a 0.57, and Random Forest a 0.56. You can really see the overfitting happen with Random Forest if you look at the data—its training accuracy was super high (somewhere between 0.83 and 0.94), but it totally dropped off on the test set with that 0.56 average.

3. Which models perform feature selection or regularization? Did they perform better than the other models?

Logistic Regression and both SVC models use regularization, while the tree ensembles (Random Forest, AdaBoost, XGBoost) naturally handle feature selection. For this dataset, the regularized models clearly came out on top. LR was first at 0.63, and SVC2 and SVC1 were right behind it at 0.62 and 0.61. The feature selection models were just okay in comparison, with AdaBoost hitting 0.58, XGBoost at 0.57, and Random Forest bringing up the rear at 0.56.

4. Which models are parametric? Did they perform better than the other models?

Our parametric models were Logistic Regression, Naive Bayes, and Linear SVC. As a group, they did really well since LR and Linear SVC were the top two models overall (0.63 and 0.62). Naive Bayes dragged the group down a bit, though, finishing with a balanced accuracy of just 0.54.

5. What model performed best? How stable was it? What does the sensitivity and specificity indicate?

Logistic Regression was the winner here, getting an average balanced accuracy of 0.63. It was fairly stable, fluctuating by about +/- 0.09 across the runs. With a sensitivity of 0.59 and specificity of 0.66, the model is pretty balanced. Instead of just blindly guessing "positive" like Naive Bayes did earlier, LR can actually tell the difference between the mild depression and non-mild depression cases, though it's slightly better at spotting the negative ones.

6. How does the best model and its performance differ across group member's tasks? Interpret.

We divided up the dataset based on depression severity (PHQ-8 scores). Grant took the no/minimal depression group (0-4), I took mild (5-9), Jackson had moderate (10-14), and Simon took moderately severe to severe (15+).

My best model for the mild group was Logistic Regression with a 0.63 balanced accuracy. Looking at the others, Grant's best was also LR but at 0.54, Jackson's was AdaBoost at 0.51, and Simon got a 0.58 with Naive Bayes.

Seeing these different results really shows how hard it is to catch depression levels just from reading text. My chunk of the data (the 5-9 scores) is especially tricky because people with mild depression often sound a lot like healthy people having a bad day. It's a blurry line.

When you look at moderate or severe cases (Jackson and Simon's data), the signs usually become a lot more obvious—like using way more negative words or giving really short, disconnected answers. Because my "mild" labels are stuck right in that gray area, the models had a harder time pulling the groups apart. Still, hitting 0.63 with Logistic Regression means it was catching at least some of those subtle hints instead of just guessing randomly.

In [ ]:
!jupyter nbconvert --to html "P2_N2_Classification.ipynb"